# Diagnostic — Entraînement OpticalCNN vs ElectronicCNN

Ce notebook entraîne les deux architectures sur les séquences générées
dans `01_generate_data.ipynb` et compare :

- la performance brute (accuracy) ;
- le nombre de paramètres (comparaison équitable) ;
- le temps d'entraînement et d'inférence ;
- la robustesse au bruit gaussien ajouté aux images de test.

Le notebook imprime les résultats sans graphique, pour faciliter
l'extraction des chiffres clés vers un rapport.


In [1]:
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path

from onn_arquS.network import OpticalCNN, ElectronicCNN
from onn_arquS.training import train_model
from onn_arquS.metrics import accuracy, confusion_matrix, delta_acc, count_parameters

torch.manual_seed(42)
np.random.seed(42)

# Parametres de simulation
INPUT_SIZE = 32
N_FRAMES = 10
N_EPOCHS = 15
BATCH_SIZE = 4
N_CLASSES = 3
CLASSES = ['nominal', 'atom_loss', 'aberration_drift']

print("=" * 70)
print("ENTRAINEMENT — OpticalCNN vs ElectronicCNN")
print("=" * 70)
print(f"Parametres : input={INPUT_SIZE}, frames={N_FRAMES}, epochs={N_EPOCHS}, batch={BATCH_SIZE}")

ENTRAINEMENT — OpticalCNN vs ElectronicCNN
Parametres : input=32, frames=10, epochs=15, batch=4


## 1. Chargement et prétraitement des données

Les séquences font `(50, 128, 128)`. On les sous-échantillonne à
`(10, 32, 32)` pour réduire le coût de calcul sur CPU :
- temporellement : 10 frames régulièrement espacées ;
- spatialement : moyenne de blocs 4×4.

La normalisation par statistiques globales évite que le classifieur
apprenne à partir de l'échelle absolue (qui varie avec les paramètres
de génération).

In [2]:
print("\n[1] Chargement des donnees")
t0 = time.time()

data_root = Path('../data')
X_list, y_list = [], []
for ci, cls in enumerate(CLASSES):
    for f in sorted((data_root / cls).glob('seq_*.npz')):
        frames = np.load(f, allow_pickle=True)['frames']
        idx_t = np.linspace(0, frames.shape[0] - 1, N_FRAMES).astype(int)
        frames = frames[idx_t].astype(np.float32)
        H, W = frames.shape[1], frames.shape[2]
        bh, bw = H // INPUT_SIZE, W // INPUT_SIZE
        frames = frames.reshape(N_FRAMES, INPUT_SIZE, bh, INPUT_SIZE, bw).mean(axis=(2, 4))
        X_list.append(frames)
        y_list.append(ci)

X = torch.tensor(np.stack(X_list), dtype=torch.float32)
y = torch.tensor(y_list, dtype=torch.long)
print(f"    X : {tuple(X.shape)}, y : {tuple(y.shape)}")
print(f"    Temps de chargement : {time.time()-t0:.1f} s")

X_mean = X.mean()
X_std = X.std()
X = (X - X_mean) / X_std
print(f"    Normalisation : mean={X_mean:.2f}, std={X_std:.2f}")


[1] Chargement des donnees
    X : (900, 10, 32, 32), y : (900,)
    Temps de chargement : 311.1 s
    Normalisation : mean=10.51, std=6.91


## 2. Split stratifié 70/15/15

In [3]:
print("\n[2] Split stratifie")
idx_train, idx_val, idx_test = [], [], []
rng = np.random.default_rng(42)
for ci in range(N_CLASSES):
    idx_c = np.where(y.numpy() == ci)[0]
    rng.shuffle(idx_c)
    n = len(idx_c)
    n_tr, n_va = int(0.7 * n), int(0.15 * n)
    idx_train.extend(idx_c[:n_tr])
    idx_val.extend(idx_c[n_tr:n_tr + n_va])
    idx_test.extend(idx_c[n_tr + n_va:])

X_train, y_train = X[idx_train], y[idx_train]
X_val, y_val = X[idx_val], y[idx_val]
X_test, y_test = X[idx_test], y[idx_test]
print(f"    Train : {len(idx_train)} | Val : {len(idx_val)} | Test : {len(idx_test)}")

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE)


[2] Split stratifie
    Train : 630 | Val : 135 | Test : 135


## 3. Architecture des modèles

In [4]:
print("\n[3] Architecture")
opt_model = OpticalCNN(input_size=INPUT_SIZE, n_layers=2, n_classes=N_CLASSES)
elec_model = ElectronicCNN(n_layers=2, n_channels=8, n_classes=N_CLASSES)
print(f"    OpticalCNN    : {count_parameters(opt_model):,} parametres")
print(f"    ElectronicCNN : {count_parameters(elec_model):,} parametres")


[3] Architecture
    OpticalCNN    : 4,227 parametres
    ElectronicCNN : 18,331 parametres


## 4. Entraînement OpticalCNN

In [5]:
print("\n[4] Entrainement OpticalCNN")
hist_opt = train_model(opt_model, train_loader, val_loader,
                       n_epochs=N_EPOCHS, lr=1e-3, patience=3, verbose=True)
print(f"    Temps total : {hist_opt['train_time']:.1f} s")


[4] Entrainement OpticalCNN
  epoch  1/15 | train 0.3983 | val 0.0666 | acc 0.985 | 10.1s
  epoch  2/15 | train 0.0752 | val 0.0210 | acc 0.985 | 19.7s
  epoch  3/15 | train 0.0421 | val 0.0086 | acc 1.000 | 30.9s
  epoch  4/15 | train 0.0329 | val 0.0057 | acc 1.000 | 40.6s
  epoch  5/15 | train 0.0106 | val 0.0101 | acc 0.993 | 50.4s
  epoch  6/15 | train 0.0093 | val 0.0057 | acc 1.000 | 60.1s
  epoch  7/15 | train 0.0093 | val 0.0016 | acc 1.000 | 69.5s
  epoch  8/15 | train 0.0104 | val 0.0007 | acc 1.000 | 79.0s
  epoch  9/15 | train 0.0116 | val 0.0013 | acc 1.000 | 88.4s
  epoch 10/15 | train 0.0034 | val 0.0003 | acc 1.000 | 97.9s
  epoch 11/15 | train 0.0036 | val 0.0006 | acc 1.000 | 107.4s
  epoch 12/15 | train 0.0079 | val 0.0003 | acc 1.000 | 117.2s
  epoch 13/15 | train 0.0107 | val 0.0022 | acc 1.000 | 126.7s
  epoch 14/15 | train 0.0045 | val 0.0005 | acc 1.000 | 136.1s
  epoch 15/15 | train 0.0037 | val 0.0001 | acc 1.000 | 145.8s
    Temps total : 145.8 s


## 5. Entraînement ElectronicCNN

In [6]:
print("\n[5] Entrainement ElectronicCNN")
hist_elec = train_model(elec_model, train_loader, val_loader,
                        n_epochs=N_EPOCHS, lr=1e-3, patience=3, verbose=True)
print(f"    Temps total : {hist_elec['train_time']:.1f} s")


[5] Entrainement ElectronicCNN
  epoch  1/15 | train 1.1023 | val 1.0964 | acc 0.333 | 0.3s
  epoch  2/15 | train 1.0924 | val 1.0895 | acc 0.333 | 0.5s
  epoch  3/15 | train 1.0793 | val 1.0748 | acc 0.667 | 0.7s
  epoch  4/15 | train 1.0581 | val 1.0547 | acc 0.417 | 1.0s
  epoch  5/15 | train 1.0260 | val 1.0348 | acc 0.333 | 1.2s
  epoch  6/15 | train 1.0187 | val 1.0309 | acc 0.333 | 1.4s
  epoch  7/15 | train 1.0073 | val 1.0237 | acc 0.333 | 1.6s
  epoch  8/15 | train 1.0053 | val 1.0194 | acc 0.333 | 1.8s
  epoch  9/15 | train 1.0020 | val 1.0129 | acc 0.333 | 2.0s
  epoch 10/15 | train 0.9920 | val 1.0034 | acc 0.417 | 2.3s
  epoch 11/15 | train 0.9803 | val 0.9992 | acc 0.417 | 2.5s
  epoch 12/15 | train 0.9692 | val 0.9882 | acc 0.417 | 2.7s
  epoch 13/15 | train 0.9637 | val 0.9800 | acc 0.417 | 2.9s
  epoch 14/15 | train 0.9527 | val 0.9667 | acc 0.417 | 3.1s
  epoch 15/15 | train 0.9363 | val 0.9568 | acc 0.417 | 3.4s
    Temps total : 3.4 s


## 6. Évaluation sur le test

In [7]:
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    t0 = time.time()
    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            preds.extend(out.argmax(1).numpy())
            labels.extend(y.numpy())
    return np.array(preds), np.array(labels), time.time() - t0

print("\n" + "=" * 70)
print("EVALUATION SUR TEST")
print("=" * 70)

pred_opt, lab, t_inf_opt = evaluate(opt_model, test_loader)
pred_elec, _, t_inf_elec = evaluate(elec_model, test_loader)

acc_opt = accuracy(lab, pred_opt)
acc_elec = accuracy(lab, pred_elec)
print(f"Accuracy OpticalCNN    : {acc_opt:.3f}")
print(f"Accuracy ElectronicCNN : {acc_elec:.3f}")
print(f"Temps d'inference (test) : Optical {t_inf_opt*1000:.0f} ms | Elec {t_inf_elec*1000:.0f} ms")

print("\nMatrice de confusion OpticalCNN :")
print(f"  {confusion_matrix(lab, pred_opt, N_CLASSES)}")
print("Matrice de confusion ElectronicCNN :")
print(f"  {confusion_matrix(lab, pred_elec, N_CLASSES)}")


EVALUATION SUR TEST
Accuracy OpticalCNN    : 0.333
Accuracy ElectronicCNN : 0.333
Temps d'inference (test) : Optical 15 ms | Elec 27 ms

Matrice de confusion OpticalCNN :
  [[3 0 2]
 [2 0 3]
 [3 0 2]]
Matrice de confusion ElectronicCNN :
  [[5 0 0]
 [5 0 0]
 [4 1 0]]


## 7. Robustesse au bruit

On ajoute un bruit gaussien d'écart-type variable aux images de test
et on mesure la dégradation de l'accuracy pour chaque modèle.

In [8]:
print("\n" + "=" * 70)
print("ROBUSTESSE AU BRUIT")
print("=" * 70)
print(f"{'sigma (ADU)':>12} | {'acc Optical':>12} | {'acc Elec':>10} | "
      f"{'delta_opt':>10} | {'delta_elec':>11}")
print("-" * 70)

for sigma in [0, 2, 5, 10, 20]:
    X_noisy = X_test + sigma * torch.randn_like(X_test)
    loader_noisy = DataLoader(TensorDataset(X_noisy, y_test), batch_size=BATCH_SIZE)
    pred_opt_n, _, _ = evaluate(opt_model, loader_noisy)
    pred_elec_n, _, _ = evaluate(elec_model, loader_noisy)
    acc_opt_n = accuracy(y_test.numpy(), pred_opt_n)
    acc_elec_n = accuracy(y_test.numpy(), pred_elec_n)
    d_opt = delta_acc(acc_opt, acc_opt_n)
    d_elec = delta_acc(acc_elec, acc_elec_n)
    print(f"{sigma:>12.1f} | {acc_opt_n:>12.3f} | {acc_elec_n:>10.3f} | "
          f"{d_opt:>10.3f} | {d_elec:>11.3f}")


ROBUSTESSE AU BRUIT
 sigma (ADU) |  acc Optical |   acc Elec |  delta_opt |  delta_elec
----------------------------------------------------------------------
         0.0 |        0.333 |      0.333 |      0.000 |       0.000
         2.0 |        0.400 |      0.333 |     -0.200 |       0.000
         5.0 |        0.333 |      0.333 |      0.000 |       0.000
        10.0 |        0.333 |      0.333 |      0.000 |       0.000
        20.0 |        0.333 |      0.333 |      0.000 |       0.000


## 8. Verdict et interprétation

In [9]:
print("\n" + "=" * 70)
print("VERDICT")
print("=" * 70)

gap = acc_elec - acc_opt
if abs(gap) < 0.05:
    print("Parite en accuracy : les deux architectures sont equivalentes.")
    print("Le reseau photonique justifie son interet sur latence/energie.")
elif gap > 0:
    print(f"Baseline electronique superieure de {gap*100:.1f} points.")
    print("Le reseau photonique a 2 couches ne suffit pas.")
    print("Piste : tester 3 couches phase-only ou 1 couche complexe.")
else:
    print(f"OpticalCNN superieur de {-gap*100:.1f} points.")
    print("Verifier l'absence d'artefact (tache triviale).")

print(f"\nTache triviale ? acc_opt={acc_opt:.3f}, acc_elec={acc_elec:.3f}")
if min(acc_opt, acc_elec) > 0.95:
    print("-> OUI. La robustesse au bruit est le vrai discriminant.")
    print("   Pour rendre la tache non triviale, reduire photons_per_atom")
    print("   ou augmenter le niveau de bruit dans la generation.")


VERDICT
Parite en accuracy : les deux architectures sont equivalentes.
Le reseau photonique justifie son interet sur latence/energie.

Tache triviale ? acc_opt=0.333, acc_elec=0.333


## 9. Conclusion

### Ce que ce notebook a mesuré
- La performance des deux architectures à paramètres comparables.
- Le coût en temps d'entraînement et d'inférence.
- La robustesse au bruit gaussien additif.

### Ce que ce notebook ne prouve pas
- Que la tâche reflète une difficulté expérimentale réelle.
- Que la robustesse au bruit gaussien est représentative du bruit
  de photons ou des aberrations.
- Que l'entraînement sur 90 séquences est suffisant (overfitting
  possible avec seulement 63 séquences d'entraînement).

### Prochaine étape
Selon le verdict :
- Parité : construire `04_evaluate_robustness.ipynb` avec un sweep
  multi-paramètres (photons, aberrations, désalignement) et un front
  de Pareto.
- Baseline supérieure : tester 3 couches ou une couche complexe.
- Tâche triviale : régénérer les données avec moins de photons par
  atome pour créer une difficulté réelle.
